In [1]:
import numpy as np

In [ ]:
# constants (blessed 2026-07-15 -- keep in sync with slides and answers.md)
m_total = 15e-6 # kg (measured: whole graphite + sail rotor = 0.015 g)
r_graphite = (3.9e-3)/2 #(2.9e-3)/2 # m (graphite radius)
w_sail = 1e-2 # m (sail width) - 1 cm; w_sail/2 is lever arm
d_sail = 50e-6 # m (sail depth) - 50 um
h_sail = 0.3e-2 # m (sail height) - 3 mm
k_b = 1.38e-23 # J/K (Boltzmann constant)
# 40 kBq, one alpha per Pb-212 decay: <p> = 0.36*p(6.1 MeV) + 0.64*p(8.8 MeV) = 1.29e-19 kg m/s
# ideal all-forward: 5.1e-15 N; /4 thin-source geometry (half into substrate, <cos theta>=1/2)
force_alpha = 1.3e-15 # N (net directed thrust)
damping_constant = 2.3e-4 # 0702 run with shorter time range fit # 2.4e-4 # Hz (1/s) post-George w/ tilt stage, 2026-07-02 spindown fit (pre-tilt was 1.3e-3)
b = 1 # 1/s (bandwidth)  -- can change to longer integration times to reduce b and thus thermal noise
rho_sail = 1.39e3 # kg/m³ (mylar density)
T = 300 # K (temperature)

In [3]:
m_sail = rho_sail * w_sail * d_sail * h_sail
m_graphite = m_total - m_sail
print(f"m_sail = {m_sail*1e6:.2f} mg, m_graphite = {m_graphite*1e6:.2f} mg")

m_sail = 2.09 mg, m_graphite = 12.92 mg


In [4]:
def inertia(m_graphite, m_sail, radius_graphite, width_sail, depth_sail): # total inertia of system
    I = (1/2 * m_graphite * radius_graphite**2) + ((1/12) * m_sail * (width_sail**2 + depth_sail**2))
    return I

In [5]:
def sigma_torque(kb, T, I, gamma, b):
    """Calculate the thermal noise torque spectral density - N*m with bandwidth; (N*m/(Hz)^1/2) - no bandwidth.
    Parameters:
    kb: float
        Boltzmann constant (J/K).
    T: float
        Temperature (K).
    I: float
        Moment of inertia (kg*m^2) of disk and sail system.
    gamma: float
        Angular damping constant (1/s).
    b: float
        Bandwidth (Hz).

    Returns:
    float
        Thermal noise torque spectral density (N*m or N*m/(Hz)^1/2 depending on including bandwidth).
    """
    return np.sqrt(4 * kb * T * I * gamma * b)

In [6]:
I = inertia(m_graphite, m_sail, r_graphite, w_sail, d_sail)
print(I)

4.1930078125e-11


In [7]:
sigma_T = sigma_torque(k_b, T, I, damping_constant, b)
print(sigma_T)

1.2637376371798856e-17


In [8]:
# signal torque: net alpha thrust acting at the sail tip (lever arm w/2)
tau_signal = w_sail/2 * force_alpha
print(f"tau_signal = {tau_signal:.2e} N*m")
print(f"SNR at b = {b} Hz: {tau_signal/sigma_T:.2f}")

tau_signal = 6.50e-18 N*m
SNR at b = 1 Hz: 0.51


In [9]:
# SNR vs integration time for the 1-minute spin-flip differential scheme.
# White thermal noise: SNR grows as sqrt(t). The Pb-212 source decays as exp(-lam*t),
# so the matched-filter SNR uses sqrt(integral of exp(-2 lam t)) and saturates at 1/(2*lam) ~ 7.7 h equivalent.
lam = np.log(2) / (10.64 * 3600)  # Pb-212 decay constant (1/s)
snr_1s = tau_signal / sigma_torque(k_b, T, I, damping_constant, 1.0)

for t_int, label in [(60, "1 min (one flip half-cycle)"), (3600, "1 hour"), (10*3600, "10 hours (~1 half-life)")]:
    snr_naive = snr_1s * np.sqrt(t_int)
    t_eff = (1 - np.exp(-2*lam*t_int)) / (2*lam)   # decay-weighted effective integration time
    snr_decay = snr_1s * np.sqrt(t_eff)
    print(f"{label:30s} SNR = {snr_decay:6.1f}   (naive sqrt(t): {snr_naive:.1f})")

print(f"{'ceiling (t -> infinity)':30s} SNR = {snr_1s*np.sqrt(1/(2*lam)):6.1f}")

1 min (one flip half-cycle)    SNR =    4.0   (naive sqrt(t): 4.0)
1 hour                         SNR =   29.9   (naive sqrt(t): 30.9)
10 hours (~1 half-life)        SNR =   73.0   (naive sqrt(t): 97.6)
ceiling (t -> infinity)        SNR =   85.5
